# Chapter 9 Tutorial — Renewal Theory

This notebook is a step-by-step tutorial for the concepts in Chapter 9: **Renewal Theory**.

The chapter starts from a simple picture:

> A phenomenon happens repeatedly. The waiting times between occurrences are i.i.d. nonnegative random variables. What can we say about the number of occurrences, long-run rates, residual lifetimes, and regenerative processes?

We will cover:

1. Renewal processes and renewal times
2. Convolution and sums of interarrival times
3. Renewal counting process
4. Renewal function and renewal measure
5. Recurrent, transient, arithmetic, and non-arithmetic renewal processes
6. Elementary renewal theorem
7. Lifetime / residual delay
8. Renewal equations
9. Key renewal theorem
10. Regenerative processes
11. Delayed renewal processes

The notebook uses simulations and plots to build intuition, and includes the main proofs and examples from the chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial, exp

rng = np.random.default_rng(7)

def ecdf_sample_path(times, t_grid=None):
    if t_grid is None:
        t_grid = np.linspace(0, times[-1] * 1.05, 400)
    counts = np.searchsorted(times, t_grid, side="right")
    return t_grid, counts

## 1. Renewal processes

A **renewal process** models repeated occurrences.

Let

\[
W_1,W_2,\ldots
\]

be independent and identically distributed nonnegative random variables. These are the **inter-renewal times** or **waiting times** between occurrences.

Define renewal epochs:

\[
S_0 = 0,\qquad S_{n+1}=S_n+W_{n+1}.
\]

So

\[
S_n = W_1+\cdots+W_n.
\]

The process

\[
S=\{S_n:n\in\mathbb N\}
\]

is called a renewal process. The \(S_n\)'s are the times at which renewals occur.

### Thinking model

Imagine a machine part. Each time it fails, it is replaced immediately by a new identical item. The lifetime of the \(n\)-th item is \(W_n\). The failure/replacement times are \(S_1,S_2,\ldots\).

In [ ]:
# Simulate a renewal process with Gamma interarrival times
n = 30
W = rng.gamma(shape=2.0, scale=1.0, size=n)
S = np.concatenate([[0], np.cumsum(W)])

plt.figure(figsize=(8, 2.8))
plt.eventplot(S, orientation="horizontal")
plt.yticks([])
plt.xlabel("time")
plt.title("Renewal epochs $S_0,S_1,S_2,\\ldots$")
plt.show()

S[:10]

## 2. Examples of renewal processes

### Example 1: Vehicle crossings

Vehicles crossing a point on a highway generate event times \(S_n\). The waiting time between vehicle \(n-1\) and vehicle \(n\) is \(W_n\).

If the \(W_n\)'s are independent and identically distributed, the crossing times form a renewal process.

### Example 2: Replacement of machine items

Suppose a part has lifetime \(U_1\). When it fails, it is replaced by another identical part with lifetime \(U_2\), and so on.

Then

\[
S_1=U_1,\quad S_2=U_1+U_2,\quad \ldots
\]

is a renewal process.

### Important nuance

If a replacement is not immediate, and there is a repair/replacement delay \(V_n\), then the cycle length may be

\[
W_n = U_n + V_n.
\]

If the pairs \((U_n,V_n)\) are i.i.d., then the cycle lengths \(W_n\) are i.i.d., and the renewal process is still valid.

## 3. Convolution and sums of interarrival times

Let \(\varphi\) be the distribution function of \(W_1\). For distribution functions \(\varphi\) and \(\psi\) on \(\mathbb R_+\), their convolution is

\[
(\varphi * \psi)(t)=\int_{[0,t]}\varphi(t-x)\,d\psi(x),\qquad t\ge 0.
\]

If \(X\sim \varphi\), \(Y\sim \psi\), and \(X,Y\) are independent, then

\[
X+Y \sim \varphi * \psi.
\]

### Proof

For \(t\ge 0\),

\[
P(X+Y\le t)
= E[P(X+Y\le t\mid Y)]
= E[\varphi(t-Y)\mathbf 1_{\{Y\le t\}}]
= \int_{[0,t]}\varphi(t-y)\,d\psi(y).
\]

So the distribution function of \(X+Y\) is \(\varphi*\psi\).

### Iterated convolution

Define

\[
\varphi^0(t)=
\begin{cases}
0,&t<0,\\
1,&t\ge 0,
\end{cases}
\]

and

\[
\varphi^{n+1}=\varphi*\varphi^n.
\]

Then \(S_n=W_1+\cdots+W_n\) has distribution function

\[
F_n=\varphi^n.
\]

In [ ]:
# Visualize sums of exponential interarrival times: Erlang/Gamma distributions
import math

def gamma_pdf(t, n, rate=1.0):
    # Erlang density for sum of n exponential(rate) variables
    return (rate**n) * (t**(n-1)) * np.exp(-rate*t) / math.factorial(n-1)

t = np.linspace(0, 15, 500)
plt.figure(figsize=(8, 4))
for n in [1, 2, 3, 5, 10]:
    plt.plot(t, gamma_pdf(t, n, rate=1.0), label=f"$S_{n}$")
plt.xlabel("t")
plt.ylabel("density")
plt.title("Distribution of renewal epochs when interarrival times are exponential")
plt.legend()
plt.show()

## 4. Renewal counting process

The number of renewals in \([0,t]\) is

\[
N_t=\sum_{n=0}^{\infty} \mathbf 1_{\{S_n\le t\}}.
\]

Since \(S_0=0\), we always have \(N_t\ge 1\). Some books count only renewals after time 0; this chapter counts the initial renewal at \(0\).

The event \(\{N_t=k\}\) means

\[
S_{k-1}\le t < S_k.
\]

Thus

\[
P(N_t=k)=P(S_{k-1}\le t)-P(S_k\le t)
= F_{k-1}(t)-F_k(t).
\]

This identity is often the easiest way to compute the distribution of \(N_t\).

In [ ]:
# Simulate N_t for Gamma interarrival renewal process
def simulate_renewal_counts(sample_interarrival, t_max=20, n_paths=5000):
    counts = []
    for _ in range(n_paths):
        s = 0.0
        count = 1  # S_0 = 0
        while True:
            s += sample_interarrival()
            if s <= t_max:
                count += 1
            else:
                break
        counts.append(count)
    return np.array(counts)

counts = simulate_renewal_counts(lambda: rng.gamma(2.0, 1.0), t_max=20, n_paths=10000)
vals, freqs = np.unique(counts, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(vals, freqs / freqs.sum())
plt.xlabel("$k$")
plt.ylabel("$P(N_{20}=k)$ estimated")
plt.title("Distribution of number of renewals in [0,20]")
plt.show()

counts.mean()

## 5. Renewal function

The expected number of renewals in \([0,t]\) is

\[
R(t)=E[N_t].
\]

Using

\[
N_t=\sum_{n=0}^{\infty}\mathbf 1_{\{S_n\le t\}},
\]

we get

\[
R(t)=\sum_{n=0}^{\infty}P(S_n\le t)
=\sum_{n=0}^{\infty}F_n(t).
\]

Equivalently,

\[
R = 1 + F + F^2 + F^3 + \cdots.
\]

This is called the **renewal function** corresponding to \(F\).

### Thinking model

\(F_n(t)\) is the probability that the \(n\)-th renewal has happened by time \(t\). Summing over \(n\) counts the expected number of renewals that have happened by \(t\).

## 6. Renewal measure / renewal operator

For a bounded measurable function \(f\) vanishing outside a finite interval, define

\[
Rf = E\left[\sum_{n=0}^{\infty}f(S_n)\right].
\]

This can be written as

\[
Rf=\int_{[0,\infty)} R(ds) f(s).
\]

So \(R\) is not only a function \(R(t)\), but also a measure/linear operator that sums over renewal epochs.

### Proof sketch

For indicator functions \(f=\mathbf 1_{[0,t]}\),

\[
Rf=E\left[\sum_{n=0}^{\infty}\mathbf 1_{\{S_n\le t\}}\right]=E[N_t]=R(t).
\]

By linearity and monotone convergence, the identity extends to bounded measurable functions with finite support.

In [ ]:
# Monte Carlo approximation of Rf for a test function f
def estimate_Rf(sample_interarrival, f, t_cut=30, n_paths=5000):
    vals = []
    for _ in range(n_paths):
        total = f(0.0)
        s = 0.0
        while s <= t_cut:
            s += sample_interarrival()
            if s <= t_cut:
                total += f(s)
        vals.append(total)
    return np.mean(vals)

f = lambda x: np.exp(-0.2*x) * (x <= 20)
estimate_Rf(lambda: rng.exponential(1.0), f, t_cut=40, n_paths=5000)

## 7. Example: Poisson process as a renewal process

If the interarrival times are exponential with rate \(\lambda\), then the renewal process is a Poisson process with an arrival at \(0\).

In that case, the expected number of renewals in \([0,t]\) is

\[
R(t)=1+\lambda t.
\]

The \(1\) is from \(S_0=0\). The remaining \(\lambda t\) is the expected number of later Poisson arrivals.

### Simulation check

In [ ]:
lam = 2.0
t_values = np.linspace(0, 10, 50)

def simulate_R_exponential(t, lam=2.0, n_paths=3000):
    counts = []
    for _ in range(n_paths):
        s = 0.0
        count = 1
        while True:
            s += rng.exponential(1/lam)
            if s <= t:
                count += 1
            else:
                break
        counts.append(count)
    return np.mean(counts)

R_est = np.array([simulate_R_exponential(t, lam=lam, n_paths=1500) for t in t_values])

plt.figure(figsize=(8, 4))
plt.plot(t_values, R_est, "o", label="simulation")
plt.plot(t_values, 1 + lam*t_values, label=r"$1+\lambda t$")
plt.xlabel("t")
plt.ylabel("$R(t)$")
plt.title("Renewal function for exponential interarrival times")
plt.legend()
plt.show()

## 8. Example: headway-constrained traffic model

Suppose a vehicle will not cross until at least \(b\) units after the previous vehicle. After that, the additional waiting time is exponential with rate \(\lambda\).

Then

\[
W=b+E,\qquad E\sim \mathrm{Exp}(\lambda).
\]

The \(n\)-th renewal epoch satisfies

\[
S_n = nb + (E_1+\cdots+E_n).
\]

Thus

\[
F_n(t)=
\begin{cases}
0,&t<nb,\\[4pt]
1-\displaystyle\sum_{k=0}^{n-1} e^{-\lambda(t-nb)}
\frac{[\lambda(t-nb)]^k}{k!},&t\ge nb.
\end{cases}
\]

Therefore

\[
R(t)=\sum_{n=0}^{\infty}F_n(t),
\]

where terms with \(nb>t\) are zero.

In [ ]:
def erlang_cdf(x, n, lam):
    if x < 0:
        return 0.0
    if n == 0:
        return 1.0
    return 1 - sum(np.exp(-lam*x) * (lam*x)**k / math.factorial(k) for k in range(n))

def delayed_exponential_R(t, b=1.0, lam=1.0):
    # R(t) = sum_n F_n(t), where F_0(t)=1
    max_n = int(t // b) + 1
    return sum(erlang_cdf(t - n*b, n, lam) for n in range(max_n + 1))

ts = np.linspace(0, 20, 200)
Rs = np.array([delayed_exponential_R(t, b=1.0, lam=1.5) for t in ts])

plt.figure(figsize=(8, 4))
plt.plot(ts, Rs)
plt.xlabel("t")
plt.ylabel("R(t)")
plt.title("Renewal function for W = fixed headway + exponential delay")
plt.show()

## 9. Recurrent and transient renewal processes

A renewal process is **recurrent** if

\[
W_n<\infty\quad \text{almost surely for every }n.
\]

It is **transient** otherwise.

Since the \(W_n\)'s are i.i.d., recurrence is equivalent to

\[
F(\infty)=\lim_{t\to\infty}F(t)=1.
\]

If

\[
F(\infty)<1,
\]

then each attempted renewal has probability \(1-F(\infty)\) of never happening. In that case, the total number of renewals is finite almost surely.

For a transient renewal process,

\[
P(N=k)= [1-F(\infty)]F(\infty)^{k-1},\qquad k=1,2,\ldots
\]

and

\[
R(\infty)=E[N]=\frac{1}{1-F(\infty)}.
\]

In [ ]:
# Simulate a transient renewal process: with probability 0.2, the next waiting time is infinite.
def sample_defective_exp(p_continue=0.8, rate=1.0):
    if rng.random() > p_continue:
        return np.inf
    return rng.exponential(1/rate)

def simulate_transient_N(p_continue=0.8, n_paths=10000):
    Ns = []
    for _ in range(n_paths):
        n = 1 # S0
        while True:
            w = sample_defective_exp(p_continue)
            if np.isinf(w):
                break
            n += 1
        Ns.append(n)
    return np.array(Ns)

Ns = simulate_transient_N(0.8)
vals, freqs = np.unique(Ns, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar(vals[:20], (freqs / freqs.sum())[:20])
plt.xlabel("total number of renewals")
plt.ylabel("probability")
plt.title("Transient renewal process: geometric total count")
plt.show()

Ns.mean(), 1/(1-0.8)

## 10. Arithmetic and non-arithmetic renewal processes

A renewal process is **periodic/arithmetic** with span \(\delta>0\) if the interarrival times take values in

\[
\{0,\delta,2\delta,\ldots\}
\]

and \(\delta\) is the largest such span.

If no such \(\delta\) exists, the renewal process is **aperiodic/non-arithmetic**.

### Examples

* If \(W\in\{1,2,3,\ldots\}\), the process is arithmetic with span \(1\).
* If \(W\in\{2,4,6,\ldots\}\), the span is \(2\).
* If \(W\) has an exponential distribution, the process is non-arithmetic.
* If \(W\) takes values \(2\) and \(3\), the span is \(1\).
* If \(W\) takes values \(\sqrt 2\) and \(3\), the process is non-arithmetic.

Arithmeticity matters because renewal epochs can only occur on a lattice, so limiting statements must respect the lattice.

## 11. Mean interarrival time

Define

\[
m = E[W_1]=\int_0^\infty [1-F(t)]\,dt.
\]

This is the expected time between renewals.

For recurrent aperiodic renewal processes, the **elementary renewal theorem** says

\[
\lim_{t\to\infty}\frac{R(t)}{t}=\frac{1}{m}.
\]

Since \(N_t\) is the number of renewals up to time \(t\),

\[
\lim_{t\to\infty}\frac{N_t}{t}=\frac{1}{m}
\]

almost surely.

### Thinking model

Long-run rate = reciprocal of average cycle length.

This is the same intuition as:

\[
\text{jobs per hour}=\frac{1}{\text{hours per job}}.
\]

In [ ]:
# Check N_t/t -> 1/E[W] for gamma interarrival times
shape, scale = 2.0, 1.5
mean_W = shape * scale
t_grid = np.linspace(1, 500, 300)

# one long sample path
W_long = rng.gamma(shape, scale, size=100000)
S_long = np.concatenate([[0], np.cumsum(W_long)])
N_grid = np.searchsorted(S_long, t_grid, side="right")

plt.figure(figsize=(8, 4))
plt.plot(t_grid, N_grid / t_grid, label=r"$N_t/t$")
plt.axhline(1/mean_W, linestyle="--", label=r"$1/E[W]$")
plt.xlabel("t")
plt.ylabel("rate")
plt.title("Long-run renewal rate")
plt.legend()
plt.show()

## 12. Transient renewal lifetime

For a transient renewal process, define

\[
L=\sup\{S_n:S_n<\infty\}.
\]

This is the time of the last renewal, also called the lifetime of the renewal process.

A useful identity is

\[
P(L\le t)=1-[1-F(\infty)]R(t).
\]

The expected lifetime is

\[
E[L]=\frac{1}{1-F(\infty)}\int_0^\infty [F(\infty)-F(t)]\,dt.
\]

### Interpretation

If the process has a chance of ending after every cycle, then the lifetime is a random sum of cycle lengths. The formula above separates:

1. how likely the process is to continue, and
2. how long each successful renewal cycle tends to last.

## 13. Example: pedestrian delay

Suppose cars pass a crossing according to a renewal process. A pedestrian arrives at time \(0\) and can cross only if the gap to the next car exceeds \(\tau\).

Let \(L\) be the waiting delay until a usable gap appears.

If the renewal process has interarrival distribution \(F\), define

\[
F_0(t)=
\begin{cases}
\varphi(t),&t\le \tau,\\
\varphi(\tau),&t>\tau.
\end{cases}
\]

Then the delay satisfies

\[
P(L\le t)=(1-F_0(\infty))R(t)=(1-\varphi(\tau))R(t).
\]

The expected delay is

\[
E[L]
=
\frac{1}{1-\varphi(\tau)}
\int_0^\tau [\varphi(\tau)-\varphi(t)]\,dt.
\]

For a Poisson traffic flow with rate \(\lambda\),

\[
\varphi(t)=1-e^{-\lambda t},
\]

so

\[
E[L]=\frac{1}{\lambda}(e^{\lambda \tau}-1)-\tau.
\]

### Intuition

For large \(\tau\), long gaps are rare. The expected waiting time grows roughly exponentially in \(\tau\).

In [ ]:
lam = 0.5
taus = np.linspace(0.01, 8, 200)
EL = (np.exp(lam * taus) - 1)/lam - taus

plt.figure(figsize=(8, 4))
plt.plot(taus, EL)
plt.xlabel(r"required gap $\tau$")
plt.ylabel(r"$E[L]$")
plt.title("Expected pedestrian delay for Poisson traffic")
plt.show()

## 14. Example: Geiger counter locked periods

A Geiger counter receives particles according to a renewal process. When a particle arrives, the counter is locked for time \(\tau\). If more particles arrive during the locked period, they are not registered, and they extend the busy period.

This is another renewal-lifetime problem. The locked period is the lifetime of a transient renewal process whose interarrival distribution is truncated at \(\tau\):

\[
W'_n=
\begin{cases}
W_n,&W_n\le \tau,\\
+\infty,&W_n>\tau.
\end{cases}
\]

So

\[
F'(t)=
\begin{cases}
F(t),&t\le \tau,\\
F(\tau),&t>\tau.
\end{cases}
\]

The distribution and expectation of the locked period follow from the transient renewal formulas.

## 15. Regenerative processes

A stochastic process \(Z=\{Z_t:t\ge 0\}\) is **regenerative** if there are random times

\[
0=S_0<S_1<S_2<\cdots
\]

such that, after each \(S_n\), the future probabilistic behavior restarts from the same law, independently of the past.

Formally, for suitable functions \(f\),

\[
E[f(Z_{S_n+t}:t\ge 0)\mid Z_u, u\le S_n]
=
E[f(Z_t:t\ge 0)].
\]

The renewal times \(S_n\) are **regeneration times**.

### Examples

* Markov chain returning to a fixed recurrent state.
* Queueing system becoming empty.
* Machine replacement process: after a replacement, the future starts fresh.
* Alternating on/off process after each full cycle.

The key idea:

> To understand the long-run behavior of \(Z\), study one cycle and then use renewal theory.

## 16. Renewal equation

Many regenerative problems lead to an equation of the form

\[
f(t)=g(t)+\int_{[0,t]} F(ds)f(t-s).
\]

In convolution notation,

\[
f=g+F*f.
\]

This is called the **renewal equation**.

The solution is

\[
f=R*g,
\]

where

\[
R=1+F+F^2+\cdots
\]

is the renewal function/measure.

### Proof

Starting from

\[
f=g+F*f,
\]

substitute recursively:

\[
f=g+F*(g+F*f)
=g+F*g+F^2*f.
\]

After \(n\) substitutions,

\[
f=g+F*g+\cdots+F^n*g+F^{n+1}*f.
\]

Under the chapter's hypotheses, \(F^{n+1}(t)\to 0\) on bounded intervals, so the remainder vanishes and

\[
f=(1+F+F^2+\cdots)*g=R*g.
\]

In [ ]:
# Solve a renewal equation numerically on a grid:
# f(t) = g(t) + ∫_0^t f(t-s) dF(s)
# Here F is exponential; discretize with density.
dt = 0.02
T = 20
grid = np.arange(0, T+dt, dt)
rate = 1.0
density = rate * np.exp(-rate * grid)

g = np.exp(-0.2 * grid)  # bounded forcing term
f = np.zeros_like(grid)
f[0] = g[0]

# crude Volterra recursion
for i in range(1, len(grid)):
    conv = np.sum(density[1:i+1] * f[i-1::-1]) * dt
    f[i] = g[i] + conv

plt.figure(figsize=(8, 4))
plt.plot(grid, g, label="g(t)")
plt.plot(grid, f, label="solution f(t)")
plt.xlabel("t")
plt.title("Numerical renewal equation solution")
plt.legend()
plt.show()

## 17. Direct Riemann integrability

The key renewal theorem needs a regularity condition on \(g\), called **direct Riemann integrability**.

Roughly:

* \(g\) should be integrable on \([0,\infty)\),
* its positive and negative oscillations on small intervals should not accumulate badly,
* and it should decay sufficiently at infinity.

Common sufficient conditions:

1. \(g\ge 0\), continuous, and vanishes outside a finite interval.
2. \(g\ge 0\), bounded and continuous, with suitable decay.
3. \(g\ge 0\), monotone non-increasing, and Riemann integrable.

This condition prevents pathological functions from breaking the limiting theorem.

## 18. Key renewal theorem

Let \(F\) be recurrent, non-arithmetic, with finite mean

\[
m=\int_0^\infty [1-F(t)]\,dt.
\]

If \(g\) is directly Riemann integrable, then

\[
\lim_{t\to\infty} R*g(t)
=
\frac{1}{m}\int_0^\infty g(s)\,ds.
\]

For arithmetic renewal processes with span \(\delta\),

\[
\lim_{n\to\infty} R*g(n\delta)
=
\frac{\delta}{m}\sum_{k=0}^{\infty}g(k\delta).
\]

### Thinking model

The renewal measure behaves asymptotically like uniform mass with density \(1/m\). Therefore convolving it with a nice function \(g\) gives:

\[
\text{long-run renewal density}\times\text{area under }g.
\]

In [ ]:
# Demonstrate key renewal theorem by simulation/approximation for Gamma interarrivals
# Estimate R*g(t) = E[sum_n g(t-S_n)] for g(s)=exp(-s)1{s>=0}.
shape, scale = 2.0, 1.0
m = shape * scale
g_func = lambda x: np.exp(-x) * (x >= 0)

def estimate_Rg_at_t(t, n_paths=4000):
    vals = []
    for _ in range(n_paths):
        s = 0.0
        total = g_func(t - 0.0)
        while s <= t:
            s += rng.gamma(shape, scale)
            if s <= t:
                total += g_func(t - s)
            else:
                break
        vals.append(total)
    return np.mean(vals)

ts = np.linspace(2, 60, 40)
Rg_est = np.array([estimate_Rg_at_t(t, n_paths=1000) for t in ts])
limit = 1/m * 1.0  # integral exp(-s) ds = 1

plt.figure(figsize=(8, 4))
plt.plot(ts, Rg_est, "o-", label=r"estimated $R*g(t)$")
plt.axhline(limit, linestyle="--", label=r"$\frac{1}{m}\int g$")
plt.xlabel("t")
plt.ylabel(r"$R*g(t)$")
plt.title("Key renewal theorem: convergence of R*g(t)")
plt.legend()
plt.show()

## 19. Refined asymptotic for the renewal function

The elementary renewal theorem gives

\[
R(t)\sim \frac{t}{m}.
\]

A sharper result in the chapter says that, under suitable conditions,

\[
\lim_{t\to\infty}\left[R(t)-\frac{t}{m}\right]
=
\frac{m^2+v^2}{2m^2},
\]

where

\[
m=E[W],
\qquad
v^2=\operatorname{Var}(W).
\]

This constant depends on both the mean and variance of the cycle length.

### Check with exponential interarrival times

For \(W\sim\mathrm{Exp}(\lambda)\),

\[
m=\frac1\lambda,\qquad v^2=\frac1{\lambda^2}.
\]

So

\[
\frac{m^2+v^2}{2m^2}=1.
\]

And since \(R(t)=1+\lambda t\),

\[
R(t)-\frac{t}{m}=1+\lambda t-\lambda t=1.
\]

So the formula matches exactly.

## 20. Regenerative-process limit theorem

Let \(Z\) be regenerative with regeneration times \(S_n\), cycle length distribution \(F\), and mean cycle length \(m\).

For a set \(A\), define

\[
K(t,A)=P(Z_t\in A,\ S_1>t).
\]

Then the renewal equation gives

\[
P(Z_t\in A)=\int_{[0,t]}R(ds)K(t-s,A).
\]

So the limiting behavior of \(Z_t\) follows from the key renewal theorem.

If \(S\) is recurrent non-arithmetic and \(m<\infty\), then

\[
\lim_{t\to\infty}P(Z_t\in A)
=
\frac{1}{m}\int_0^\infty K(s,A)\,ds,
\]

provided \(K(\cdot,A)\) is directly Riemann integrable.

### Interpretation

The long-run probability of being in \(A\) is:

\[
\frac{\text{expected amount of time spent in }A\text{ during one cycle}}
{\text{expected cycle length}}.
\]

This is one of the most useful ideas in applied stochastic processes.

## 21. Example: age since last renewal

Let

\[
Z_t=t-S_{N_t}
\]

be the **age** of the renewal process at time \(t\): the time elapsed since the last renewal.

For \(A=(y,\infty)\),

\[
K(t,A)=P(S_1>t,\ Z_t>y)=P(W_1>t,\ t>y).
\]

Thus, for recurrent non-arithmetic renewal processes,

\[
\lim_{t\to\infty}P(Z_t>y)
=
\frac{1}{m}\int_y^\infty [1-F(x)]\,dx.
\]

This is the classical limiting age distribution.

### Important intuition

If you observe a renewal process at a random large time, you are more likely to land in a long interval than in a short interval. This is the inspection paradox.

In [ ]:
# Simulate limiting age distribution for Gamma interarrivals
shape, scale = 2.0, 1.0
m = shape * scale

def simulate_age_at_large_t(T=1000, n_paths=5000):
    ages = []
    for _ in range(n_paths):
        s = 0.0
        last = 0.0
        while s <= T:
            last = s
            s += rng.gamma(shape, scale)
        ages.append(T - last)
    return np.array(ages)

ages = simulate_age_at_large_t()
y_grid = np.linspace(0, 10, 100)
surv_est = np.array([(ages > y).mean() for y in y_grid])

# theoretical survival: (1/m) ∫_y^∞ survival_W(x) dx.
# Approximate by Monte Carlo tail integral over grid.
from scipy.stats import gamma
surv_theory = np.array([
    (1/m) * np.trapz(1 - gamma.cdf(np.linspace(y, 50, 2000), a=shape, scale=scale),
                     np.linspace(y, 50, 2000))
    for y in y_grid
])

plt.figure(figsize=(8, 4))
plt.plot(y_grid, surv_est, label="simulation")
plt.plot(y_grid, surv_theory, label="theory")
plt.xlabel("y")
plt.ylabel(r"$P(age>y)$")
plt.title("Limiting age distribution")
plt.legend()
plt.show()

## 22. Example: machine replacement process

Let

\[
Z_t=
\begin{cases}
1,&\text{machine is working or being replaced at time }t,\\
0,&\text{otherwise}
\end{cases}
\]

or more simply, let \(Z_t=1\) if the current item is being replaced/working in a certain phase.

If a cycle consists of:

* lifetime \(U\),
* replacement time \(V\),

then the total cycle length is

\[
W=U+V,
\]

and \(m=E[W]=E[U]+E[V]\).

The long-run fraction of time spent in the “working” phase is

\[
\frac{E[U]}{E[U]+E[V]}.
\]

This is exactly the regenerative-process ratio:

\[
\frac{\text{expected reward in one cycle}}{\text{expected cycle length}}.
\]

## 23. Markov chains and Markov processes as regenerative processes

### Discrete-time Markov chains

If \(X_n\) is a Markov chain and \(j\) is recurrent, then successive visits to \(j\) form a renewal process. The cycle lengths are return times to \(j\).

The regenerative theorem recovers the Markov-chain limiting result:

\[
\lim_{n\to\infty}P(X_n=j)=\frac{1}{m(j)},
\]

where \(m(j)\) is the mean recurrence time of state \(j\).

### Continuous-time Markov processes

For a Markov process, successive entrance times into a recurrent state \(j\) also form a regenerative process. The renewal-theoretic limit gives:

\[
\lim_{t\to\infty}P(Z_t=j)
=
\frac{1}{m(j)\lambda(j)},
\]

where:

* \(m(j)\) is the mean recurrence time in jump-count/cycle terms,
* \(1/\lambda(j)\) is the expected holding time in state \(j\).

This connects renewal theory back to Markov-process stationary behavior.

## 24. Delayed renewal processes

A **delayed renewal process** allows the first renewal interval to have a different distribution.

Let \(S_0\ge 0\) have distribution \(G\). Then after that,

\[
S_n-S_0
\]

is an ordinary renewal process with interarrival distribution \(F\).

So

\[
P(S_n\le t)=G*F^n(t),\qquad t\ge 0.
\]

This is useful when the process did not start exactly at a renewal time.

### Why delayed renewal matters

When observing a system “in the wild,” time \(0\) is usually not a renewal epoch. You might start observing a machine halfway through its current lifetime, or a queue after it has already been running.

The delayed-renewal framework corrects for this initial offset.

## 25. Summary table

| Concept | Formula | Meaning |
|---|---:|---|
| Renewal epochs | \(S_n=W_1+\cdots+W_n\) | Times of repeated occurrences |
| Counting process | \(N_t=\sum_n 1_{\{S_n\le t\}}\) | Number of renewals by time \(t\) |
| Distribution of \(N_t\) | \(P(N_t=k)=F_{k-1}(t)-F_k(t)\) | Count via adjacent renewal epochs |
| Renewal function | \(R(t)=E[N_t]=\sum_n F_n(t)\) | Expected renewal count |
| Renewal operator | \(Rf=E[\sum_n f(S_n)]\) | Sum reward over renewal epochs |
| Mean cycle length | \(m=E[W]\) | Average time between renewals |
| Long-run rate | \(N_t/t\to 1/m\) | Renewal frequency |
| Renewal equation | \(f=g+F*f\) | Regenerative decomposition |
| Solution | \(f=R*g\) | Iterate over cycles |
| Key renewal theorem | \(R*g(t)\to m^{-1}\int g\) | Renewal measure becomes uniform |
| Regenerative limit | \(\lim P(Z_t\in A)=m^{-1}\int K(s,A)ds\) | Long-run fraction of cycle |

## 26. Mental models

### Renewal process

A clock resets after every event. The next cycle is independent of the past and statistically identical to previous cycles.

### Renewal function

Add up the probabilities that each possible renewal has already happened:

\[
R(t)=P(S_0\le t)+P(S_1\le t)+P(S_2\le t)+\cdots.
\]

### Renewal equation

A process either:

1. does something before the first renewal, or
2. renews and starts over.

That gives:

\[
f=g+F*f.
\]

### Key renewal theorem

Far from the start, renewal epochs appear at average density \(1/m\). So a nice local test function \(g\) sees approximately:

\[
\frac{1}{m}\times\text{area under }g.
\]

### Regeneration

Complicated processes become manageable when viewed cycle-by-cycle. The long-run fraction of time in a condition is usually:

\[
\frac{\text{expected time in condition per cycle}}{\text{expected cycle length}}.
\]